[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-05-ray-train-distributed.ipynb#scrollTo=11223344)

---
# Day 5 · Ray Train — Distributed Model Training with PyTorch and scikit-learn
**certified-journeys / ray-certified** · Ray for Distributed Python · Practice Badge

> **Goal for today:** Launch distributed PyTorch and scikit-learn training jobs with Ray Train, using `TorchTrainer`, `ScalingConfig`, and `XGBoostTrainer` — all without manual process management.


In [ ]:
%pip install -q 'ray[train]' torch scikit-learn xgboost


## Step 1 · Ray Train Overview

**Ray Train** is a distributed training library that wraps popular frameworks (PyTorch, TensorFlow, Hugging Face, XGBoost, scikit-learn) behind a unified API.

Core concepts:
- **Trainer** — framework-specific entry point (`TorchTrainer`, `XGBoostTrainer`, `SklearnTrainer`).
- **`train_loop_per_worker`** — the function each distributed worker executes.
- **`ScalingConfig`** — declares how many workers, whether to use GPUs, and per-worker resources.
- **`ray.train.report()`** — sends metrics from a worker back to the driver.
- **`Result`** — object returned by `trainer.fit()`; contains metrics, checkpoints, and errors.

```
TorchTrainer(train_loop, ScalingConfig)  →  trainer.fit()  →  Result
```

📖 https://docs.ray.io/en/latest/train/train.html


In [ ]:
import ray

ray.init(ignore_reinit_error=True)

print("Ray version:", ray.__version__)
print("Available resources:", ray.cluster_resources())


### What just happened?

- **`ray.init()`** starts a local cluster; on a multi-node cluster replace with `ray.init(address='auto')`.
- Ray Train workers are standard Ray tasks/actors — they use the same scheduling primitives.
- The driver process (this notebook) coordinates workers via the GCS; workers execute the training loop independently.
- All inter-worker communication for gradient synchronisation goes through `torch.distributed` (NCCL on GPU, Gloo on CPU).


## Step 2 · ScalingConfig — Controlling Parallelism

**`ScalingConfig`** is the single object that controls resource allocation for a training job:

| Parameter | Type | Effect |
|---|---|---|
| `num_workers` | int | Number of distributed training processes |
| `use_gpu` | bool | Allocate one GPU per worker (requires CUDA) |
| `resources_per_worker` | dict | Fine-grained control: `{"CPU": 2, "memory": 2e9}` |
| `trainer_resources` | dict | Resources for the driver/coordinator actor |

For Colab: `num_workers=1, use_gpu=False` is the safe default.

📖 https://docs.ray.io/en/latest/train/api/doc/ray.train.ScalingConfig.html


In [ ]:
from ray.train import ScalingConfig

# Colab-safe: 1 CPU worker, no GPU
scaling_config = ScalingConfig(
    num_workers=1,
    use_gpu=False,
    resources_per_worker={"CPU": 1},
)

print("ScalingConfig:", scaling_config)

# Production example (not executed — comment shows multi-GPU setup):
# scaling_config_gpu = ScalingConfig(
#     num_workers=4,
#     use_gpu=True,
#     resources_per_worker={"GPU": 1, "CPU": 4},
# )


### What just happened?

- **`ScalingConfig`** is a declarative spec — no cluster calls happen yet.
- `num_workers=1` is correct for Colab: single-machine, single-process training that still exercises the full Ray Train code path.
- In production, increase `num_workers` to match available GPUs across your cluster — Ray handles placement automatically.
- **`resources_per_worker`** lets you co-locate CPU workers on the same machine or spread them for fault tolerance.


## Step 3 · TorchTrainer — 2-Layer MLP on Iris

**`TorchTrainer`** runs your `train_loop_per_worker` function on each worker. Inside the loop:

1. **`ray.train.torch.prepare_model(model)`** — wraps the model in `DistributedDataParallel` (DDP). Call it once, right after model creation.
2. **`ray.train.torch.prepare_data_loader(loader)`** — adds the correct `DistributedSampler` to ensure each worker gets a unique shard.
3. **`ray.train.report(metrics={})`** — streams metrics to the driver after each epoch.

> ⚠️ Never call `prepare_model()` inside the training loop — it re-wraps on every iteration, causing silent gradient errors.

📖 https://docs.ray.io/en/latest/train/api/doc/ray.train.torch.TorchTrainer.html


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# ── Prepare Iris data once on the driver ──────────────────────────────────
iris = load_iris()
X, y = iris.data.astype(np.float32), iris.target

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to tensors — will be serialised into the worker closure
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

print(f"Train: {X_train_t.shape}  Test: {X_test_t.shape}")


In [ ]:
import ray.train
import ray.train.torch
from ray.train.torch import TorchTrainer

# ── Define the training loop ───────────────────────────────────────────────
# This function runs on EACH worker. All imports must be inside or at module level.
def train_loop_per_worker(config: dict):
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import ray.train.torch

    # Hyperparameters passed via config dict
    lr      = config.get("lr", 0.01)
    epochs  = config.get("epochs", 10)
    hidden  = config.get("hidden", 32)

    # Retrieve tensors baked into the closure
    X_tr = config["X_train"]
    y_tr = config["y_train"]

    # ── Model: 2-layer MLP ─────────────────────────────────────────────
    model = nn.Sequential(
        nn.Linear(4, hidden),
        nn.ReLU(),
        nn.Linear(hidden, 3),  # 3 Iris classes
    )

    # CRITICAL: call prepare_model() BEFORE the training loop
    model = ray.train.torch.prepare_model(model)

    # ── DataLoader ─────────────────────────────────────────────────────
    dataset = TensorDataset(X_tr, y_tr)
    loader  = DataLoader(dataset, batch_size=32, shuffle=True)
    # prepare_data_loader adds DistributedSampler for multi-worker jobs
    loader = ray.train.torch.prepare_data_loader(loader)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # ── Training loop ──────────────────────────────────────────────────
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)

        # Report metrics to the driver after every epoch
        ray.train.report(metrics={"epoch": epoch, "loss": round(avg_loss, 4)})

print("train_loop_per_worker defined — ready to pass to TorchTrainer.")


In [ ]:
# ── Launch TorchTrainer ────────────────────────────────────────────────────
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "lr":      0.01,
        "epochs":  8,
        "hidden":  32,
        # Pass tensors via config — Ray serialises them with cloudpickle
        "X_train": X_train_t,
        "y_train": y_train_t,
    },
    scaling_config=ScalingConfig(
        num_workers=1,
        use_gpu=False,
        resources_per_worker={"CPU": 1},
    ),
)

result = trainer.fit()   # blocks until all workers finish

print("\n── Training complete ──")
print("Metrics history:", result.metrics_dataframe[["epoch", "loss"]].to_string(index=False))


### What just happened?

- **`TorchTrainer`** spawned one worker Ray actor, sent it the `train_loop_per_worker` function, and blocked until it finished.
- **`prepare_model()`** wrapped the model in `DistributedDataParallel` — with `num_workers=1` this is a no-op, but the code is ready to scale.
- **`ray.train.report()`** streams metrics to the driver after each epoch; `result.metrics_dataframe` collects them all.
- **`Result`** contains: `.metrics_dataframe` (all reported metrics), `.checkpoint` (last checkpoint), `.error` (if training failed).


## Step 4 · Inspecting the Result Object

The **`Result`** object is the central artifact returned by any Ray Train `fit()` call. Knowing its structure is essential for production pipelines.

| Attribute | Type | Content |
|---|---|---|
| `result.metrics` | dict | Metrics from the **last** reported epoch |
| `result.metrics_dataframe` | DataFrame | All reported metrics across epochs |
| `result.checkpoint` | `Checkpoint` or `None` | Latest saved checkpoint |
| `result.error` | `Exception` or `None` | Non-None if training raised an exception |
| `result.config` | dict | The `train_loop_config` used |


In [ ]:
# Inspect the Result object in detail

print("Last reported metrics:", result.metrics)
print()

# Full metrics table
df_metrics = result.metrics_dataframe
print("All epochs:")
print(df_metrics[["epoch", "loss"]].to_string(index=False))
print()

# Checkpoint (None here because we didn't save one in the training loop)
print("Checkpoint:", result.checkpoint)

# Error — None means training succeeded
print("Error:", result.error)

# Confirm loss trend is decreasing
losses = df_metrics["loss"].tolist()
print(f"\nLoss: first={losses[0]:.4f}  last={losses[-1]:.4f}  improving={losses[-1] < losses[0]}")


### What just happened?

- **`result.metrics`** is a snapshot of the last `ray.train.report()` call — useful for quick checks.
- **`result.metrics_dataframe`** lets you plot training curves or compare runs programmatically.
- **`result.checkpoint`** is `None` here because we did not call `ray.train.report(checkpoint=...)`. In production, pass a `ray.train.Checkpoint` object to preserve model weights.
- **`result.error`** being `None` confirms all workers exited cleanly — always check this in automated pipelines.


## Step 5 · XGBoostTrainer on a Ray Dataset

**`XGBoostTrainer`** (in `ray.train.xgboost`) trains an XGBoost model in a distributed fashion. It accepts a Ray Dataset directly — no manual train/test split needed in the API.

Key differences from `TorchTrainer`:
- **No `train_loop_per_worker`** — you pass XGBoost params directly.
- The dataset is passed via `datasets={"train": ds_train, "valid": ds_valid}`.
- **`label_column`** specifies the target column name in the Dataset.

📖 https://docs.ray.io/en/latest/train/api/doc/ray.train.sklearn.SklearnTrainer.html


In [ ]:
import ray.data
from ray.train.xgboost import XGBoostTrainer
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# ── Build Ray Datasets from Iris ───────────────────────────────────────────
iris = load_iris()
df_iris = pd.DataFrame(
    iris.data,
    columns=["sepal_length", "sepal_width", "petal_length", "petal_width"]
)
df_iris["label"] = iris.target

train_df, valid_df = train_test_split(df_iris, test_size=0.2, random_state=42, stratify=df_iris["label"])

ds_train = ray.data.from_pandas(train_df)
ds_valid = ray.data.from_pandas(valid_df)

print("Train dataset:", ds_train.schema())
print(f"Train rows: {ds_train.count()}   Valid rows: {ds_valid.count()}")


In [ ]:
# ── Launch XGBoostTrainer ──────────────────────────────────────────────────
xgb_trainer = XGBoostTrainer(
    scaling_config=ScalingConfig(
        num_workers=1,       # XGBoost distributed = 1 worker per CPU node
        use_gpu=False,
    ),
    label_column="label",   # column in the Dataset that holds the target
    params={
        "objective":        "multi:softprob",
        "num_class":        3,
        "max_depth":        4,
        "learning_rate":    0.1,
        "n_estimators":     50,
        "eval_metric":      "mlogloss",
    },
    datasets={"train": ds_train, "valid": ds_valid},
    num_boost_round=30,
)

xgb_result = xgb_trainer.fit()

print("\n── XGBoost training complete ──")
print("Last metrics:", xgb_result.metrics)


### What just happened?

- **`XGBoostTrainer`** accepts Ray Datasets natively — it handles sharding to XGBoost workers internally.
- **`label_column`** tells the trainer which column is the target; all other columns are features.
- The `datasets` dict can include `"train"` and `"valid"` — XGBoost uses `"valid"` for early stopping if configured.
- `xgb_result.metrics` includes the final validation loss — use `xgb_result.metrics_dataframe` to inspect per-round metrics.


## Step 6 · Saving and Loading Checkpoints

**Checkpoints** let you resume training, do inference from the best epoch, or deploy a trained model. In Ray Train:

- Inside the training loop: `ray.train.report(metrics={...}, checkpoint=ray.train.Checkpoint.from_dict({"model": state_dict}))`
- After training: `result.checkpoint.to_dict()["model"]` to retrieve weights.

We add checkpointing to a minimal re-run of the MLP loop to show the full pattern.


In [ ]:
import ray.train
from ray.train import Checkpoint

def train_loop_with_checkpoint(config: dict):
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import ray.train
    import ray.train.torch
    import tempfile, os

    X_tr = config["X_train"]
    y_tr = config["y_train"]
    epochs = config.get("epochs", 5)

    model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
    model = ray.train.torch.prepare_model(model)  # DDP wrap — before training loop

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    loader = ray.train.torch.prepare_data_loader(loader)

    opt  = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item()
        avg = total / len(loader)

        # Save checkpoint to a temp dir each epoch
        with tempfile.TemporaryDirectory() as tmpdir:
            ckpt_path = os.path.join(tmpdir, "model.pt")
            # Unwrap DDP to get the raw module state dict
            state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            torch.save(state, ckpt_path)
            checkpoint = ray.train.Checkpoint.from_directory(tmpdir)
            ray.train.report(
                metrics={"epoch": epoch, "loss": round(avg, 4)},
                checkpoint=checkpoint,
            )

trainer_ckpt = TorchTrainer(
    train_loop_per_worker=train_loop_with_checkpoint,
    train_loop_config={"epochs": 5, "X_train": X_train_t, "y_train": y_train_t},
    scaling_config=ScalingConfig(num_workers=1, use_gpu=False),
)

result_ckpt = trainer_ckpt.fit()

print("Last metrics:", result_ckpt.metrics)
print("Checkpoint:", result_ckpt.checkpoint)


### What just happened?

- **`ray.train.Checkpoint.from_directory(tmpdir)`** creates a checkpoint from any directory of files — flexible for PyTorch, TensorFlow, or plain JSON.
- **`model.module.state_dict()`** unwraps the `DistributedDataParallel` wrapper to get the underlying model weights.
- **`result_ckpt.checkpoint`** holds the *last* checkpoint; to keep the *best* checkpoint, use `RunConfig(checkpoint_config=CheckpointConfig(num_to_keep=1, checkpoint_score_attribute='loss', checkpoint_score_order='min'))`.
- Loading: `result_ckpt.checkpoint.to_directory(path)` restores files; then `torch.load(path + '/model.pt')`.


## Step 7 · Training Multiple Configurations with a Loop

Before Ray Tune (Day 6), you can still sweep hyperparameters manually using a Python `for` loop over `ScalingConfig` or `train_loop_config` variants. This gives you a taste of why Tune is so valuable — it automates and parallelises this loop.


In [ ]:
import pandas as pd

# Simple grid sweep: 2 learning rates × 2 hidden sizes = 4 runs
results_log = []

for lr in [0.01, 0.05]:
    for hidden in [16, 32]:

        def make_loop(lr_val, hidden_val):
            """Factory that bakes lr and hidden into the closure."""
            def loop(config):
                import torch, torch.nn as nn
                from torch.utils.data import DataLoader, TensorDataset
                import ray.train.torch

                X_tr = config["X_train"]
                y_tr = config["y_train"]

                model = nn.Sequential(
                    nn.Linear(4, hidden_val), nn.ReLU(), nn.Linear(hidden_val, 3)
                )
                model = ray.train.torch.prepare_model(model)

                loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
                loader = ray.train.torch.prepare_data_loader(loader)

                opt, loss_fn = torch.optim.Adam(model.parameters(), lr=lr_val), nn.CrossEntropyLoss()

                for epoch in range(5):
                    model.train()
                    total = sum(
                        (lambda loss: (loss.backward(), opt.step(), opt.zero_grad(), loss.item()))[3]
                        for xb, yb in loader
                        for loss in [loss_fn(model(xb), yb)]
                    )
                    ray.train.report({"epoch": epoch + 1, "loss": round(total / len(loader), 4)})
            return loop

        t = TorchTrainer(
            train_loop_per_worker=make_loop(lr, hidden),
            train_loop_config={"X_train": X_train_t, "y_train": y_train_t},
            scaling_config=ScalingConfig(num_workers=1, use_gpu=False),
        )
        r = t.fit()
        results_log.append({"lr": lr, "hidden": hidden, "final_loss": r.metrics["loss"]})

sweep_df = pd.DataFrame(results_log).sort_values("final_loss")
print("\nGrid sweep results (sorted by loss):")
print(sweep_df.to_string(index=False))


### What just happened?

- The manual sweep runs 4 sequential training jobs — each a full Ray Train lifecycle.
- **This is exactly the motivation for Ray Tune** (Day 6): Tune parallelises these trials automatically and terminates bad ones early.
- The closure factory pattern (`make_loop(lr, hidden)`) is necessary to avoid late-binding bugs in Python closures inside loops.
- The best `(lr, hidden)` combination should generalise best to the validation set — confirmed in Day 6 with proper HPO.


In [ ]:
# ── Challenge ──────────────────────────────────────────────────────────────
# Challenge: Adapt the TorchTrainer to train on the Wine dataset instead of Iris.
#   1. Load sklearn.datasets.load_wine()
#   2. Standardise features with StandardScaler
#   3. Create a 3-layer MLP (input → 64 → 32 → 3 classes)
#   4. Train for 10 epochs and report loss + accuracy each epoch
#      Hint: accuracy = (preds == labels).float().mean().item()
#   5. Print the final accuracy from result.metrics

# Your solution here

# def wine_train_loop(config):
#     ...

# wine_trainer = TorchTrainer(...)
# wine_result = wine_trainer.fit()
# print("Final accuracy:", wine_result.metrics.get("accuracy"))


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `TorchTrainer` | Runs `train_loop_per_worker` on each Ray worker; returns a `Result` object |
| `prepare_model()` | Wraps model in DDP — call once, before the training loop, never inside it |
| `prepare_data_loader()` | Adds `DistributedSampler` so each worker gets a unique shard |
| `ray.train.report()` | Streams `metrics={}` and optional `checkpoint=` to the driver each epoch |
| `ScalingConfig` | Declares `num_workers`, `use_gpu`, `resources_per_worker` — declarative, no cluster calls |
| `Result` | Contains `.metrics`, `.metrics_dataframe`, `.checkpoint`, `.error` |
| `XGBoostTrainer` | Accepts Ray Datasets directly; no `train_loop_per_worker` needed |
| `Checkpoint` | Portable artifact; use `.from_directory()` to save and `.to_directory()` to restore |

> **Tip:** `ray.train.torch.prepare_model()` wraps your model in `DistributedDataParallel` automatically. Call it after model instantiation but before the training loop — never inside the loop.

---
## What's next
**Day 6** → Ray Tune — Hyperparameter Search at Scale. You'll define trainable functions, build search spaces with `grid_search` and `loguniform`, and use `ASHAScheduler` + `OptunaSearch` to find the best hyperparameters without manual grid sweeps.

Mark Day 5 complete in your [tracker](../index.html).
